# justounetsimple_opt on the FPGA

The optimized kernel: the same quantized 2D-JustoUNet as `justounetsimple`, but
pipelined, unrolled and with its four blocks running at the same time (DATAFLOW).
It takes a whole batch of 32 x 32 patches per start instead of one.

Cloud / land / sea classification of a HYPSO-2 capture: the capture is cut into
32 x 32 patches, each is preprocessed here (10 bands dropped, z-score as in
training), the FPGA returns 3 scores per pixel. Highest score = class:
**0 cloud, 1 land, 2 sea**.

Next to this notebook: `justounetsimple_opt.bit` + `.hwh`, `xjustounetsimple_opt_hw.h`,
`mu_sd.txt`, and an image zip made on the PC with
`python3 tools/justounetsimple_image.py prepare ... --out aegean_unet` and `fpga_zip aegean_unet`
(the same image works for both kernels). Run the cells from the top.

## 1. Load the bitstream and give the kernel its DDR buffers

In [ ]:
import json, re, time, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

overlay = Overlay('justounetsimple_opt.bit')   # also sets the FPGA clock from the .hwh
kernel = overlay.kernel                         # the HLS block, named 'kernel' in the block design

header = Path('xjustounetsimple_opt_hw.h').read_text()
def register(name):
    return int(re.search(rf'_ADDR_{name}\s+(0x[0-9a-fA-F]+)', header).group(1), 16)
AP_CTRL, DIN, DOUT, N = register('AP_CTRL'), register('DIN_DATA'), register('DOUT_DATA'), register('N_DATA')

# Up to BATCH patches per start. The kernel reads each patch 16 bands at a time,
# plane by plane: [7][32][32][16] (112 padded channels: 110 real + 2 zero).
# It writes 32 x 32 x 3 scores per patch.
BATCH = 64
patches_in = allocate(shape=(BATCH, 7, 32, 32, 16), dtype=np.float32)
scores_out = allocate(shape=(BATCH, 32, 32, 3), dtype=np.float32)
for offset, buffer in ((DIN, patches_in), (DOUT, scores_out)):
    kernel.write(offset, buffer.physical_address & 0xFFFFFFFF)
    kernel.write(offset + 4, buffer.physical_address >> 32)
print('kernel idle:', bool(kernel.read(AP_CTRL) & 0x4))

## 2. The functions: preprocess, a batch of patches, a whole image

In [ ]:
# Training's preprocessing: keep 110 of the 120 bands, then (x - mean) / std.
mu, sd = [np.array(a.replace(',', ' ').split(), dtype=np.float64)
          for a in re.findall(r'\[([^\]]*)\]', Path('mu_sd.txt').read_text())]
KEPT = [b for b in range(120) if b not in (0, 1, 2, 3, 4, 5, 6, 7, 118, 119)]
MEAN, INV_STD = mu.astype(np.float32), (1 / sd).astype(np.float32)

def preprocess(raw):
    return (raw[..., KEPT].astype(np.float32) - MEAN) * INV_STD

def run_patches(x, timeout=60):
    # k preprocessed patches (k x 32 x 32 x 110, k <= BATCH) -> k x 32 x 32 x 3 scores, in one start.
    k = len(x)
    xp = np.pad(x, ((0, 0), (0, 0), (0, 0), (0, 2)))      # 110 -> 112 real bands, zero padded
    patches_in[:k] = xp.reshape(k, 32, 32, 7, 16).transpose(0, 3, 1, 2, 4)
    patches_in.flush()                     # into DDR, not just the CPU cache
    kernel.write(N, k)
    kernel.write(AP_CTRL, 1)               # start
    deadline = time.monotonic() + timeout
    while not kernel.read(AP_CTRL) & 0x2:  # wait for done
        if time.monotonic() > deadline:
            raise TimeoutError('kernel did not finish')
    scores_out.invalidate()                # read what the kernel wrote
    return np.array(scores_out[:k])

def pad(cube):
    # Grow the image to whole 32 x 32 patches by repeating its last row and column.
    h, w, _ = cube.shape
    return np.pad(cube, ((0, -h % 32), (0, -w % 32), (0, 0)), mode='edge')

def classify(cube):
    # Raw cube (H x W x 120) -> scores (H x W x 3).
    h, w, _ = cube.shape
    padded = pad(cube)
    rows, cols = padded.shape[0] // 32, padded.shape[1] // 32
    # all patches in raster order: (rows * cols) x 32 x 32 x 120
    tiles = padded.reshape(rows, 32, cols, 32, -1).transpose(0, 2, 1, 3, 4).reshape(rows * cols, 32, 32, -1)
    scores = np.empty((len(tiles), 32, 32, 3), dtype=np.float32)
    for i in range(0, len(tiles), BATCH):
        scores[i:i + BATCH] = run_patches(preprocess(tiles[i:i + BATCH]))
        print(f'  {min(i + BATCH, len(tiles))}/{len(tiles)} patches', flush=True)
    scores = scores.reshape(rows, cols, 32, 32, 3).transpose(0, 2, 1, 3, 4).reshape(rows * 32, cols * 32, 3)
    return scores[:h, :w]

## 3. Load the image

In [ ]:
IMAGE = 'aegean_unet.zip'      # or an already unpacked folder
folder = Path(IMAGE)
if folder.suffix == '.zip':
    with zipfile.ZipFile(folder) as z:
        z.extractall('.')
    folder = Path(folder.stem)
cube = np.load(folder / 'cube.npy')
reference = np.load(folder / 'reference_scores.npy')    # the numpy model's scores, from the PC
meta = json.loads((folder / 'meta.json').read_text())
print(f'{cube.shape[0]} x {cube.shape[1]} pixels, {-(-cube.shape[0] // 32) * -(-cube.shape[1] // 32)} patches')

## 4. Self-test: the first patch against the PC's answer

In [ ]:
first = run_patches(preprocess(pad(cube)[None, :32, :32]))[0]
want = reference[:32, :32]
error = np.abs(first[:want.shape[0], :want.shape[1]] - want).max()
print('PASS' if error < 1e-3 * (1 + np.abs(want).max()) else 'FAIL', f'worst error {error:.2e}')

## 5. The whole image

In [ ]:
start = time.monotonic()
scores = classify(cube)
seconds = time.monotonic() - start
print(f'{seconds:.2f} s for {cube.shape[0]} x {cube.shape[1]} pixels')
np.save(folder / 'fpga_scores.npy', scores)

## 6. Check it: against the PC's answer and against the labels

In [ ]:
classes, ref_classes = scores.argmax(-1), reference.argmax(-1)
close = np.abs(scores - reference) <= 1e-3 + 1e-3 * np.abs(reference)
print(f'scores within tolerance of the PC: {close.all(-1).mean():.2%}, same class: {(classes == ref_classes).mean():.2%}')
labels = np.load(folder / 'labels.npy') if (folder / 'labels.npy').exists() else None
if labels is not None:
    known = labels < 3
    print(f'accuracy against the labels: {(classes[known] == labels[known]).mean():.2%}')
    for c, name in enumerate(('cloud', 'land', 'sea')):
        union = ((classes == c) | (labels == c))[known].sum()
        iou = ((classes == c) & (labels == c))[known].sum() / union if union else float('nan')
        print(f'  IoU {name}: {iou:.3f}')

In [ ]:
COLORS = np.array([[235, 235, 235], [60, 140, 60], [30, 90, 180], [128, 128, 128]], dtype=np.uint8)
rgb = cube[:, :, meta['rgb_bands']].astype(np.float64)
lo, hi = np.percentile(rgb, [2, 98])
panels = {'picture': np.clip((rgb - lo) / (hi - lo), 0, 1),
          'FPGA classes': COLORS[classes],
          'differs from PC': ~close.all(-1)}
if labels is not None:
    panels['labels'] = COLORS[np.minimum(labels, 3)]
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4))
for ax, (title, img) in zip(axes, panels.items()):
    ax.imshow(img, cmap='gray_r' if img.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')
plt.show()     # cloud white, land green, sea blue, gray = unlabeled

## 7. Speed

The FPGA alone (one full batch, data already in DDR), and the whole image above
(including the preprocessing and copying in Python).

In [ ]:
x = np.random.default_rng(0).uniform(-3, 3, (BATCH, 32, 32, 110)).astype(np.float32)
run_patches(x)                         # puts a full batch in DDR and sets N
start = time.monotonic()
kernel.write(AP_CTRL, 1)
while not kernel.read(AP_CTRL) & 0x2:
    pass
fpga = time.monotonic() - start
patches = -(-cube.shape[0] // 32) * -(-cube.shape[1] // 32)
print(f'FPGA alone:  {1000 * fpga / BATCH:.2f} ms per patch ({BATCH} patches per start)')
print(f'whole image: {1000 * seconds / patches:.2f} ms per patch, {seconds:.2f} s in all')